In [8]:
# ============================================================
# SECTION 1a: INSTALL — Hugging Face datasets library

%pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# ============================================================
# SECTION 1b: LOAD MMLU — pull the dataset and filter to our 8 subjects
# ============================================================

from datasets import load_dataset

# Load the MMLU test split -- "all" pulls every subject, we filter next
print("Loading MMLU dataset (this may take a minute on first run)...")
mmlu_full = load_dataset("cais/mmlu", "all", split="test")
print(f"Loaded {len(mmlu_full)} total questions across all MMLU subjects")

# 8 subjects
YOUR_SUBJECTS = [
    "clinical_knowledge", "professional_medicine",      # Medical
    "international_law", "professional_law",            # Legal
    "econometrics", "professional_accounting",           # Financial
    "high_school_mathematics", "college_mathematics"     # Math
]

# Filter down to just  8 subjects
mmlu_filtered = mmlu_full.filter(lambda row: row["subject"] in YOUR_SUBJECTS)
print(f"\nFiltered to your 8 subjects: {len(mmlu_filtered)} questions total")

# Show how many questions are available per subject -- confirms each has
import pandas as pd
subject_counts = pd.Series(mmlu_filtered["subject"]).value_counts()
print("\nQuestions available per subject:")
print(subject_counts[YOUR_SUBJECTS])   # ordered to match your list above

Loading MMLU dataset (this may take a minute on first run)...


README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\datasets--cais--mmlu. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.50MB            

...

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Loaded 14042 total questions across all MMLU subjects


Filter:   0%|          | 0/14042 [00:00<?, ? examples/s]


Filtered to your 8 subjects: 2958 questions total

Questions available per subject:
clinical_knowledge          265
professional_medicine       272
international_law           121
professional_law           1534
econometrics                114
professional_accounting     282
high_school_mathematics     270
college_mathematics         100
Name: count, dtype: int64


In [10]:
# ============================================================
# SECTION 1c: SAVE the full filtered dataset (all 8 subjects, 2958 questions)
# to a local file, BEFORE sampling down to 320
# ============================================================

import json

# Convert the Hugging Face dataset object into a plain list of dictionaries,
# which is easy to save/load and doesn't depend on the datasets library later
mmlu_filtered_list = [dict(row) for row in mmlu_filtered]

with open("mmlu_filtered_full.jsonl", "w") as f:
    for row in mmlu_filtered_list:
        f.write(json.dumps(row) + "\n")

print(f"Saved {len(mmlu_filtered_list)} questions (all 8 subjects) to mmlu_filtered_full.jsonl")

# Quick peek at what one question actually looks like structurally
print("\nExample question structure:")
print(json.dumps(mmlu_filtered_list[0], indent=2))

Saved 2958 questions (all 8 subjects) to mmlu_filtered_full.jsonl

Example question structure:
{
  "question": "What size of cannula would you use in a patient who needed a rapid blood transfusion (as of 2020 medical knowledge)?",
  "subject": "clinical_knowledge",
  "choices": [
    "18 gauge.",
    "20 gauge.",
    "22 gauge.",
    "24 gauge."
  ],
  "answer": 0
}


In [12]:
# ============================================================
# SECTION 1e: SAMPLE 40 questions per subject, fixed seed, reproducible
# ⚠️ Depends on Section 9c (mmlu_filtered_list) already being run.
#
# CRITICAL DESIGN REQUIREMENTS being enforced here (from proposal):
# - Exactly 40 questions per subject, 8 subjects = 320 total
# - Sampled ONCE with a FIXED seed -- must be reproducible if re-run
# - This set is LOCKED after this point: reused unchanged across all
#   Rounds, all Strategies, and (ideally) both models (GPT + Gemini)
# ============================================================

import random

# One fixed seed, chosen once
SAMPLING_SEED = 42

# IMPORTANT: using Python's built-in `random` module here, NOT hash().
# random.Random(seed) is genuinely deterministic across machines/sessions
# for a given seed -- unlike hash(), which is randomized per-process for
# strings. This is safe and correctly reproducible
sampler = random.Random(SAMPLING_SEED)

sampled_questions = []   # will hold our final locked 320 questions

for subject in YOUR_SUBJECTS:
    # Get every question belonging to just this one subject
    subject_pool = [q for q in mmlu_filtered_list if q["subject"] == subject]

    # Confirm we actually have at least 40 to sample from (safety check --
    # we already confirmed this via subject_counts earlier, but re-checking
    # here protects against silent errors if this cell is ever run alone)
    assert len(subject_pool) >= 40, f"Subject {subject} has fewer than 40 questions!"

    # Randomly sample exactly 40, WITHOUT replacement (no question repeats),
    # using our seeded sampler -- same seed will always pick the same 40
    subject_sample = sampler.sample(subject_pool, 40)

    # Add this subject's 40 questions to our overall running list
    sampled_questions.extend(subject_sample)

# Sanity check: we should have exactly 320 questions total (40 x 8)
print(f"Total sampled: {len(sampled_questions)} questions (expected 320)")

# Sanity check: confirm exactly 40 per subject, not skewed
from collections import Counter
counts = Counter(q["subject"] for q in sampled_questions)
print("\nPer-subject counts in the sample:")
for subject in YOUR_SUBJECTS:
    print(f"  {subject}: {counts[subject]}")

Total sampled: 320 questions (expected 320)

Per-subject counts in the sample:
  clinical_knowledge: 40
  professional_medicine: 40
  international_law: 40
  professional_law: 40
  econometrics: 40
  professional_accounting: 40
  high_school_mathematics: 40
  college_mathematics: 40


In [13]:
# ============================================================
# SECTION 1f: INSPECT one sampled question's raw structure BEFORE we
# assume its format -- confirms whether 'answer' is an index or letter,
# and whether 'choices' is a list, before we write conversion code.
# ============================================================

print("Raw structure of the first sampled question -- INSPECT before assuming format:")
print(json.dumps(sampled_questions[0], indent=2))

Raw structure of the first sampled question -- INSPECT before assuming format:
{
  "question": "The cardiac cycle consists of the phases:",
  "subject": "clinical_knowledge",
  "choices": [
    "systole, diastole, and rest.",
    "contraction, relaxation, and rest.",
    "diastole and systole.",
    "diastole, systole, and contraction."
  ],
  "answer": 2
}


In [14]:
# ============================================================
# SECTION 1g: CONVERT — turn one MMLU question into run_trial's expected format
# ⚠️ Depends on Section 9e (sampled_questions) already being run.
#
# MMLU gives us: {"question": "...", "choices": [4 strings], "answer": 0-3 (int)}
# run_trial needs: question_text (with "A) ... B) ... C) ... D) ..." formatted in),
#                   correct_answer_letter ("A"/"B"/"C"/"D")
# ============================================================

def convert_mmlu_question(mmlu_row):
    """
    Converts one raw MMLU question dict into the format run_trial expects.

    mmlu_row: one dict from sampled_questions, e.g.
              {"question": "...", "choices": [...], "answer": 2, "subject": "..."}

    Returns a dict with:
        question_text          -- the question plus "A) ... B) ... C) ... D) ..."
        correct_answer_letter   -- "A", "B", "C", or "D"
        options                 -- dict {"A": choice0, "B": choice1, ...} -- needed
                                    later for select_target_wrong_answer, which
                                    expects this exact dict shape
        subject                 -- passed through unchanged, for building question_id
    """

    # MMLU's 4 choices, in order -- index 0 through 3
    choices = mmlu_row["choices"]

    # Map index 0/1/2/3 to letter A/B/C/D
    letters = ["A", "B", "C", "D"]

    # Build the options dict, e.g. {"A": "systole, diastole...", "B": "...", ...}
    # This exact shape is required by select_target_wrong_answer, which we
    # already built and tested earlier.
    options = {letters[i]: choices[i] for i in range(4)}

    # Build the full question text with lettered choices appended,
    # exactly matching the format used in all our 6a-6d test questions
    question_text = mmlu_row["question"] + "\n" + "\n".join(
        f"{letters[i]}) {choices[i]}" for i in range(4)
    )

    # Convert the integer answer index into its corresponding letter
    correct_answer_letter = letters[mmlu_row["answer"]]

    return {
        "question_text": question_text,
        "correct_answer_letter": correct_answer_letter,
        "options": options,
        "subject": mmlu_row["subject"]
    }


# ---- TEST: convert the same question we just inspected, confirm it's correct ----
converted = convert_mmlu_question(sampled_questions[0])
print("Converted question:")
print(converted["question_text"])
print(f"\nCorrect answer letter: {converted['correct_answer_letter']}")
print(f"Options dict: {converted['options']}")

Converted question:
The cardiac cycle consists of the phases:
A) systole, diastole, and rest.
B) contraction, relaxation, and rest.
C) diastole and systole.
D) diastole, systole, and contraction.

Correct answer letter: C
Options dict: {'A': 'systole, diastole, and rest.', 'B': 'contraction, relaxation, and rest.', 'C': 'diastole and systole.', 'D': 'diastole, systole, and contraction.'}


In [16]:
# ============================================================
# SECTION 1h-verify (UPDATED): REPRODUCIBILITY CHECK — rerun sampling,
# confirm identical results, AND print sample questions to visually inspect
# ============================================================

sampler_check = random.Random(SAMPLING_SEED)   # same fixed seed as before

sampled_questions_CHECK = []
for subject in YOUR_SUBJECTS:
    subject_pool = [q for q in mmlu_filtered_list if q["subject"] == subject]
    subject_sample = sampler_check.sample(subject_pool, 40)
    sampled_questions_CHECK.extend(subject_sample)

# Compare the two full samples for exact equality
identical = (sampled_questions == sampled_questions_CHECK)
print(f"Samples are identical: {identical}\n")

# ---- Print 2 example questions from each, side by side, so you can
#      visually confirm they're the same questions, in the same order ----
print("=" * 70)
print("VISUAL CHECK -- first 2 questions of each subject, ORIGINAL vs REPRODUCED")
print("=" * 70)

for subject in YOUR_SUBJECTS:
    original_subject_qs = [q for q in sampled_questions if q["subject"] == subject]
    check_subject_qs = [q for q in sampled_questions_CHECK if q["subject"] == subject]

    print(f"\n--- {subject} ---")
    for i in range(2):
        print(f"  [Original  {i+1}]: {original_subject_qs[i]['question'][:70]}...")
        print(f"  [Reproduced {i+1}]: {check_subject_qs[i]['question'][:70]}...")
        match = "✓ MATCH" if original_subject_qs[i] == check_subject_qs[i] else "✗ MISMATCH"
        print(f"  {match}")

if not identical:
    print("\n⚠️ Full samples do NOT match despite the same seed -- investigate before saving.")
else:
    print("\n✓ Confirmed: reproducibility holds. Safe to proceed to Section 9h.")

Samples are identical: True

VISUAL CHECK -- first 2 questions of each subject, ORIGINAL vs REPRODUCED

--- clinical_knowledge ---
  [Original  1]: The cardiac cycle consists of the phases:...
  [Reproduced 1]: The cardiac cycle consists of the phases:...
  ✓ MATCH
  [Original  2]: Which of the following statements is false?...
  [Reproduced 2]: Which of the following statements is false?...
  ✓ MATCH

--- professional_medicine ---
  [Original  1]: A couple comes for preconceptional genetic counseling because they bot...
  [Reproduced 1]: A couple comes for preconceptional genetic counseling because they bot...
  ✓ MATCH
  [Original  2]: Over 1 year, a study is conducted to assess the antileukemic activity ...
  [Reproduced 2]: Over 1 year, a study is conducted to assess the antileukemic activity ...
  ✓ MATCH

--- international_law ---
  [Original  1]: What is meant by an international organisation's implied powers?...
  [Reproduced 1]: What is meant by an international organisation's

In [17]:
# ============================================================
# SECTION 1h-preview: PREVIEW question_ids only, WITHOUT saving anything
# This does NOT write any file -- pure preview, so we can inspect the
# ID scheme before committing to it in the real 9h save step.
# ============================================================

preview_ids = []

for mmlu_row in sampled_questions:
    subject = mmlu_row["subject"]

    # Count how many of this subject we've already assigned an ID to
    subject_index = sum(1 for qid in preview_ids if qid.startswith(subject + "_")) + 1
    question_id = f"{subject}_{subject_index:03d}"

    preview_ids.append(question_id)

# Print the first 3 and last 3 IDs for EACH subject, so we can see the
# numbering start and end clearly
print("PREVIEW of question_ids (first 3 and last 3 per subject):\n")
for subject in YOUR_SUBJECTS:
    subject_ids = [qid for qid in preview_ids if qid.startswith(subject + "_")]
    print(f"{subject}:")
    print(f"  First 3: {subject_ids[:3]}")
    print(f"  Last 3:  {subject_ids[-3:]}")
    print()

PREVIEW of question_ids (first 3 and last 3 per subject):

clinical_knowledge:
  First 3: ['clinical_knowledge_001', 'clinical_knowledge_002', 'clinical_knowledge_003']
  Last 3:  ['clinical_knowledge_038', 'clinical_knowledge_039', 'clinical_knowledge_040']

professional_medicine:
  First 3: ['professional_medicine_001', 'professional_medicine_002', 'professional_medicine_003']
  Last 3:  ['professional_medicine_038', 'professional_medicine_039', 'professional_medicine_040']

international_law:
  First 3: ['international_law_001', 'international_law_002', 'international_law_003']
  Last 3:  ['international_law_038', 'international_law_039', 'international_law_040']

professional_law:
  First 3: ['professional_law_001', 'professional_law_002', 'professional_law_003']
  Last 3:  ['professional_law_038', 'professional_law_039', 'professional_law_040']

econometrics:
  First 3: ['econometrics_001', 'econometrics_002', 'econometrics_003']
  Last 3:  ['econometrics_038', 'econometrics_039',

In [18]:
# ============================================================
# SECTION 1h-preview2: PREVIEW one FULL formatted question per subject,
# exactly as it will be saved -- question_id, question_text, correct
# answer letter, and options dict.
# This does NOT save anything -- pure preview before the real save.
# ============================================================

preview_ids = []   # rebuild IDs the same way, to pair with converted content

for i, mmlu_row in enumerate(sampled_questions):
    subject = mmlu_row["subject"]
    subject_index = sum(1 for qid in preview_ids if qid.startswith(subject + "_")) + 1
    question_id = f"{subject}_{subject_index:03d}"
    preview_ids.append(question_id)

    # Only show the FIRST question of each subject for this preview
    if subject_index == 1:
        converted = convert_mmlu_question(mmlu_row)

        print("=" * 70)
        print(f"question_id: {question_id}")
        print(f"subject: {subject}")
        print("-" * 70)
        print("question_text:")
        print(converted["question_text"])
        print(f"\ncorrect_answer_letter: {converted['correct_answer_letter']}")
        print(f"\noptions: {converted['options']}")
        print()

question_id: clinical_knowledge_001
subject: clinical_knowledge
----------------------------------------------------------------------
question_text:
The cardiac cycle consists of the phases:
A) systole, diastole, and rest.
B) contraction, relaxation, and rest.
C) diastole and systole.
D) diastole, systole, and contraction.

correct_answer_letter: C

options: {'A': 'systole, diastole, and rest.', 'B': 'contraction, relaxation, and rest.', 'C': 'diastole and systole.', 'D': 'diastole, systole, and contraction.'}

question_id: professional_medicine_001
subject: professional_medicine
----------------------------------------------------------------------
question_text:
A couple comes for preconceptional genetic counseling because they both have a family history of α-thalassemia. The woman has a minimally decreased hemoglobin concentration. Genetic studies show a single gene deletion. The man has microcytic anemia and a two-gene deletion. If the two-gene deletion is in trans (one deletion o

In [20]:
# ============================================================
# SECTION 1h (FINAL, with full summary): SAVE the locked 320-question dataset
# ============================================================

locked_dataset = []

for mmlu_row in sampled_questions:
    converted = convert_mmlu_question(mmlu_row)
    subject = converted["subject"]

    subject_index = sum(1 for q in locked_dataset if q["subject"] == subject) + 1
    question_id = f"{subject}_{subject_index:03d}"

    locked_dataset.append({
        "question_id": question_id,
        "subject": subject,
        "question_text": converted["question_text"],
        "correct_answer_letter": converted["correct_answer_letter"],
        "options": converted["options"]
    })

with open("locked_320_questions.jsonl", "w") as f:
    for q in locked_dataset:
        f.write(json.dumps(q) + "\n")

# ---- Full summary, topic/category-wise ----
print("=" * 60)
print("LOCKED DATASET SAVED — locked_320_questions.jsonl")
print("=" * 60)

print(f"\nTotal questions: {len(locked_dataset)}")
print(f"Sampling seed: {SAMPLING_SEED} (fixed, reproducible)")
print(f"Reproducibility: verified identical across independent re-runs")

print(f"\nSubjects: {len(YOUR_SUBJECTS)} (4 categories x 2 subjects each)")
print(f"\n{'Category':<12} {'Subject':<28} {'Questions'}")
print("-" * 55)

categories = {
    "Medical": ["clinical_knowledge", "professional_medicine"],
    "Legal": ["international_law", "professional_law"],
    "Financial": ["econometrics", "professional_accounting"],
    "Math": ["high_school_mathematics", "college_mathematics"]
}

grand_total = 0
for category, subjects in categories.items():
    category_total = 0
    for subject in subjects:
        count = sum(1 for q in locked_dataset if q["subject"] == subject)
        print(f"{category:<12} {subject:<28} {count}")
        category_total += count
    print(f"{'':12} {'-- ' + category + ' subtotal':<28} {category_total}")
    print()
    grand_total += category_total

print("-" * 55)
print(f"{'TOTAL':<41} {grand_total}")

print(f"\nID scheme: <subject>_<3-digit number>, e.g. clinical_knowledge_001 to clinical_knowledge_040")
print(f"First question_id overall: {locked_dataset[0]['question_id']}")
print(f"Last question_id overall: {locked_dataset[-1]['question_id']}")

LOCKED DATASET SAVED — locked_320_questions.jsonl

Total questions: 320
Sampling seed: 42 (fixed, reproducible)
Reproducibility: verified identical across independent re-runs

Subjects: 8 (4 categories x 2 subjects each)

Category     Subject                      Questions
-------------------------------------------------------
Medical      clinical_knowledge           40
Medical      professional_medicine        40
             -- Medical subtotal          80

Legal        international_law            40
Legal        professional_law             40
             -- Legal subtotal            80

Financial    econometrics                 40
Financial    professional_accounting      40
             -- Financial subtotal        80

Math         high_school_mathematics      40
Math         college_mathematics          40
             -- Math subtotal             80

-------------------------------------------------------
TOTAL                                     320

ID scheme: <subject>_<3

In [2]:
from datetime import datetime

print("Dataset download record:")
print(f"  Dataset: cais/mmlu")
print(f"  Downloaded on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Subjects: ['clinical_knowledge', 'professional_medicine', 'international_law', 'professional_law', 'econometrics', 'professional_accounting', 'high_school_mathematics', 'college_mathematics']")
print(f"  Sampling seed: 42")
print(f"  Total questions sampled: 320")

Dataset download record:
  Dataset: cais/mmlu
  Downloaded on: 2026-08-30 08:16:47
  Subjects: ['clinical_knowledge', 'professional_medicine', 'international_law', 'professional_law', 'econometrics', 'professional_accounting', 'high_school_mathematics', 'college_mathematics']
  Sampling seed: 42
  Total questions sampled: 320
